# Backfill the Qdrant collection

Rebuilds a Qdrant collection from a raw Reddit export of r/PESU.

The write itself is `backfill()` in `services/db/scripts/populate_db.py` — the same function the
command line runs. This notebook prepares its inputs and calls it.

## Where it runs

- A local kernel in a checkout.
- A remote kernel from an editor, such as VS Code attached to a Colab runtime.
- This file uploaded on its own to Colab or Kaggle. With no checkout around it, it clones the
  repo.

## What you have to supply

Both are gitignored, so neither arrives with a clone:

1. The raw export, as `dump/r_PESU_posts.jsonl` and `dump/r_PESU_comments.jsonl`. Section 5
   builds everything else from these.
2. `QDRANT_URL`, `QDRANT_API_KEY` and `QDRANT_COLLECTION`, either in a `.env` at the repo root or
   already set in the environment.

Getting them onto a hosted kernel is up to the host: `colab upload`, a Kaggle dataset, a mounted
Drive, a browser upload.

On a local kernel, install the CUDA build of torch first:
`uv sync --extra api --extra db --no-group cpu --group gpu`

Clear cell outputs before committing this file.


## 1. Locate the code and the corpus


In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/pesu-dev/ask-pesu.git"


def find_repo() -> Path | None:
    """Walk up for a checkout, identified by the file both services contract on."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "conf" / "collection.yaml").is_file():
            return base
    return None


repo = find_repo()
if repo is None:
    # No checkout, so this is a hosted kernel. The repo is public; clone it
    # for the code rather than pasting the write path into a cell.
    repo = Path("/content/ask-pesu")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    print(f"cloned {REPO_URL} -> {repo}")
else:
    print(f"local checkout: {repo}")

DB = repo / "services" / "db"
sys.path.insert(0, str(DB))
print("populate_db.py:", (DB / "scripts" / "populate_db.py").is_file())


cloned https://github.com/pesu-dev/ask-pesu.git -> /content/ask-pesu
populate_db.py: True


In [2]:
# The two raw JSONL files are the input. `processed/` holds one JSON file per
# post and is built from them in section 5.
DUMP = repo / "dump"
PROCESSED = DUMP / "processed"
COMPLETED = DUMP / "completed"
POSTS = DUMP / "r_PESU_posts.jsonl"
COMMENTS = DUMP / "r_PESU_comments.jsonl"

have_raw = POSTS.is_file() and COMMENTS.is_file()
if not (have_raw or PROCESSED.is_dir() or COMPLETED.is_dir()):
    DUMP.mkdir(parents=True, exist_ok=True)
    raise SystemExit(
        f"Nothing to work from at {DUMP}.\n\n"
        f"Put the Reddit export there -- {POSTS.name} and {COMMENTS.name}. The export is\n"
        "gitignored, so it never arrives with a clone and has to come from the machine that has\n"
        "it, by whatever route the host offers: colab upload, a Kaggle dataset, a mounted Drive,\n"
        "a browser upload.\n\n"
        "Send the raw files. Section 5 builds everything else from them.\n"
    )
print("dump:", DUMP)
print("  raw export:", have_raw)
print("  processed/:", PROCESSED.is_dir())


SystemExit: Nothing to work from at /content/ask-pesu/dump.

Put the Reddit export there -- r_PESU_posts.jsonl and r_PESU_comments.jsonl. The export is
gitignored, so it never arrives with a clone and has to come from the machine that has
it, by whatever route the host offers: colab upload, a Kaggle dataset, a mounted Drive,
a browser upload.

Send the raw files. Section 5 builds everything else from them.


/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 2. What machine is this?


Embedding dominates the runtime, so check that a GPU is actually in use. The default `cpu`
dependency group pins the CPU build of torch, which is what the Spaces need and not what this
does.


In [ ]:
print("python:", sys.version.split()[0])
print("cores :", os.cpu_count())
try:
    import torch

    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"gpu   : {name} ({vram:.0f} GB)")
    else:
        print("gpu   : NONE -- embedding will fall back to the CPU, which is far slower.")
        print("        Local fix: uv sync --extra api --extra db --no-group cpu --group gpu")
        print("        Colab fix: pick a GPU runtime.")
except ImportError:
    print("gpu   : torch not installed yet; the next cell handles it.")


## 3. Dependencies


In [ ]:
import importlib.util

# Packages the write path imports that a hosted kernel may not have. Versions
# come from requirements.txt, so this matches what the images install.
NEEDED = {
    "qdrant_client": "qdrant-client",
    "langchain_qdrant": "langchain-qdrant",
    "langchain_huggingface": "langchain-huggingface",
    "fastembed": "fastembed",
    "anytree": "anytree",
    "praw": "praw",
    "dotenv": "python-dotenv",
    "yaml": "pyyaml",
    "sentence_transformers": "sentence-transformers",
}
pins = {}
for line in (repo / "requirements.txt").read_text().splitlines():
    if "==" in line and not line.lstrip().startswith("#"):
        name, _, version = line.partition("==")
        pins[name.strip()] = version.split()[0].strip()

missing = [f"{dist}=={pins[dist]}" if dist in pins else dist for mod, dist in NEEDED.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", " ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("done -- restart the kernel if an import below still fails")
else:
    print("all present")


## 4. Settings and credentials


Reads `.env` from the repo root, or uses the environment if something has already set these.

The collection name is printed rather than checked here — `backfill()` validates it against
`conf/collection.yaml` before writing. Read it before running section 7.


In [ ]:
from dotenv import load_dotenv

if not load_dotenv(repo / ".env") and not os.environ.get("QDRANT_URL"):
    raise SystemExit(
        f"No {repo / '.env'}, and QDRANT_URL is not set. Send that file up, or set QDRANT_URL, "
        "QDRANT_API_KEY and QDRANT_COLLECTION in the environment -- which is what a Kaggle or "
        "Colab secret does for you."
    )

# Both bound memory and request size, not the vectors produced, so they are
# safe to tune per machine. See DEFAULT_ENCODE_BATCH_SIZE and DEFAULT_BATCH_SIZE
# in populate_db.py for how the shipped values are chosen.
ENCODE_BATCH_SIZE = 64
BATCH_SIZE = 256

# Which collection section 7 will write.
print("collection:", os.environ.get("QDRANT_COLLECTION"))
print("credentials loaded:", bool(os.environ.get("QDRANT_URL")) and bool(os.environ.get("QDRANT_API_KEY")))


## 5. Build the processed corpus


Builds `processed/` from the raw export: one JSON file per post, which is what the write path
reads. CPU-only and parallel across cores, so it does not use the GPU.

Skipped if the work is already done. Note that a completed write leaves `processed/` empty:
`populate_db.py` moves each file to `completed/` once it is stored, so an interrupted run can
resume from where it stopped. This cell moves them back.


In [ ]:
def rebuild() -> None:
    """Derive the processed corpus from the raw JSONL export."""
    subprocess.run(
        [sys.executable, "scripts/generate_processed_data.py",
         "--posts", str(POSTS), "--comments", str(COMMENTS), "--output-dir", str(PROCESSED)],
        cwd=DB, check=True,
    )


pending = list(PROCESSED.glob("*.json")) if PROCESSED.is_dir() else []
retired = list(COMPLETED.glob("*.json")) if COMPLETED.is_dir() else []

if pending:
    print(f"already built: {len(pending)} files in {PROCESSED}")
elif retired:
    # A previous run stored everything and moved it to completed/. Move it
    # back instead of rebuilding from the raw export.
    print(f"restoring {len(retired)} files from a completed run")
    PROCESSED.mkdir(parents=True, exist_ok=True)
    for path in retired:
        path.rename(PROCESSED / path.name)
elif POSTS.is_file() and COMMENTS.is_file():
    rebuild()
else:
    raise SystemExit(f"Nothing to build from: no files in {PROCESSED}, {COMPLETED}, and no raw export.")


In [ ]:
import json

files = sorted(PROCESSED.glob("*.json"))
documents = sum(len(json.loads(p.read_text()).get("comments", [])) for p in files)
print(f"{len(files)} files | {documents} documents to write")


## 6. Dry run


Validates the collection against `conf/collection.yaml` and checks every payload against the
contract. Builds no embedding model and writes nothing.


In [ ]:
sys.path.insert(0, str(DB / "scripts"))
from populate_db import backfill  # noqa: E402

# The same function the command line runs.
assert backfill(PROCESSED, COMPLETED, dry_run=True) == 0, "dry run failed -- fix this before writing"


## 7. Write


Point ids are a UUIDv5 of the root comment id, so this is an upsert: re-running after an
interruption repeats work but never duplicates a document.

On a remote kernel the output stream can drop while the kernel keeps running. If that happens,
check the collection's point count from elsewhere rather than assuming the run died.


In [ ]:
import time

started = time.perf_counter()
exit_code = backfill(
    PROCESSED,
    COMPLETED,
    encode_batch_size=ENCODE_BATCH_SIZE,
    batch_size=BATCH_SIZE,
)
print(f"\nbackfill returned {exit_code} in {time.perf_counter() - started:.0f}s")


## 8. Verify


Reads points back and checks them against the contract. The point count on its own is not
enough: an upsert over an existing collection can leave the count unchanged.


In [ ]:
from qdrant_client import QdrantClient

sys.path.insert(0, str(DB))
from app import contract as contract_mod  # noqa: E402

contract = contract_mod.load()
client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])

count = client.get_collection(contract.name).points_count
print(f"{contract.name}: {count} points (expected {documents})")

sample, _ = client.scroll(contract.name, limit=50, with_payload=True, with_vectors=True)
contracted = set(contract.metadata)
bad = [p.id for p in sample if contracted - set(p.payload.get("metadata", {}))]
vectors = {name for p in sample for name in (p.vector if isinstance(p.vector, dict) else {})}
print(f"sampled {len(sample)} | missing contracted keys: {len(bad)} | vectors present: {sorted(vectors)}")
if count != documents:
    print("NOTE: count differs from the document total -- expected only if the collection held other data.")
